# Section 7 · Governance 2 — AgentCore Policy: control what the agent *does* (Groq edition)

> **Groq edition.** Same lab as `02_agentcore_policy_cedar.ipynb`; only the agent's model runs on Groq. AgentCore Gateway, the Cedar policy engine and the Lambda tool are AWS services, so you still need AWS credentials — but no Bedrock model access. Set `GROQ_API_KEY` in your environment or a `.env` file.

AgentCore Policy intercepts every tool call that goes through an **AgentCore Gateway** and checks it against **Cedar** rules *before* the tool runs. The rules are deterministic — no prompt, however clever, changes the outcome.

```
Agent ──MCP──► AgentCore Gateway ──► Policy engine (Cedar) ──allow──► Lambda: process_refund
                                            │
                                            └──deny──► error returned to the agent
```

You will:
1. Deploy a `process_refund` tool as a Lambda function
2. Put it behind an AgentCore Gateway
3. Show that a $50,000 refund goes through (no policy yet)
4. Create a policy engine with two Cedar policies: *allow refunds* + *forbid refunds over $1,000*
5. Attach it to the gateway in `ENFORCE` mode
6. Show that $500 is allowed and $50,000 is denied — for direct calls and for the agent
7. Clean up

**Prerequisites:** AWS credentials with permissions for `bedrock-agentcore:*` (gateway + policy), `lambda:*` on the demo function, and `iam:CreateRole/PutRolePolicy/AttachRolePolicy/PassRole/Delete*`.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import io
import json
import os
import time
import uuid
import zipfile

import boto3

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
iam = session.client("iam")
lambda_client = session.client("lambda")
control = session.client("bedrock-agentcore-control")

suffix = uuid.uuid4().hex[:6]
FUNCTION_NAME = f"refund-processor-{suffix}"
LAMBDA_ROLE = f"refund-processor-role-{suffix}"
GATEWAY_ROLE = f"AgentCorePolicyGatewayRole-{suffix}"
GATEWAY_NAME = f"refund-gateway-{suffix}"
TARGET_NAME = "RefundTarget"
ENGINE_NAME = f"RefundPolicyEngine_{suffix}"

print(f"Account {account_id} | Region {REGION} | suffix {suffix}")

## Step 1: The refund tool as a Lambda function

In [ ]:
LAMBDA_CODE = '''
import json

def lambda_handler(event, context):
    # The gateway passes the tool arguments as the event
    tool = context.client_context.custom.get("bedrockAgentCoreToolName", "unknown")
    amount = event.get("amount")
    customer_id = event.get("customer_id")
    reason = event.get("reason", "")
    print(f"[{tool}] refund {amount} for {customer_id}: {reason}")
    return {"status": "PROCESSED", "customer_id": customer_id, "amount": amount,
            "message": f"Refund of ${amount} processed for customer {customer_id}"}
'''

trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
lambda_role_arn = iam.create_role(RoleName=LAMBDA_ROLE, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
iam.attach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
time.sleep(10)  # IAM propagation

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("lambda_function.py", LAMBDA_CODE)

function_arn = lambda_client.create_function(
    FunctionName=FUNCTION_NAME,
    Runtime="python3.12",
    Role=lambda_role_arn,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": buf.getvalue()},
    Timeout=15,
)["FunctionArn"]
lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION_NAME)
print("Lambda ready:", function_arn)

## Step 2: Gateway role and gateway

The gateway role may invoke the Lambda function and consult the policy engine (`GetPolicyEngine`, `AuthorizeAction`, `PartiallyAuthorizeActions`).

The gateway uses `authorizerType="NONE"` so the demo client needs no token; it is deleted at the end. In production use `CUSTOM_JWT` or `AWS_IAM` — that is also what lets Cedar rules reference *who* is calling (see `cedar-policies/03_*` and `04_*`).

In [ ]:
gateway_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": account_id},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"},
        },
    }],
}
gateway_permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": function_arn},
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetGateway",
         "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"},
        {"Effect": "Allow",
         "Action": ["bedrock-agentcore:GetPolicyEngine", "bedrock-agentcore:AuthorizeAction",
                    "bedrock-agentcore:PartiallyAuthorizeActions"],
         "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:policy-engine/*",
                      f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"]},
    ],
}
gateway_role_arn = iam.create_role(RoleName=GATEWAY_ROLE, AssumeRolePolicyDocument=json.dumps(gateway_trust))["Role"]["Arn"]
iam.put_role_policy(RoleName=GATEWAY_ROLE, PolicyName="GatewayPermissions", PolicyDocument=json.dumps(gateway_permissions))
time.sleep(10)


def wait_for(get_fn, ready=("READY",), failed=("FAILED", "UPDATE_UNSUCCESSFUL", "SYNCHRONIZE_UNSUCCESSFUL"), label=""):
    while True:
        status = get_fn()["status"]
        print(f"  {label} status: {status}")
        if status in ready:
            return
        if status in failed:
            raise RuntimeError(f"{label} failed: {get_fn()}")
        time.sleep(5)


gateway = control.create_gateway(
    name=GATEWAY_NAME,
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="NONE",
    description="Refund tools protected by AgentCore Policy",
)
GATEWAY_ID, GATEWAY_ARN = gateway["gatewayId"], gateway["gatewayArn"]
wait_for(lambda: control.get_gateway(gatewayIdentifier=GATEWAY_ID), label="Gateway")
GATEWAY_URL = control.get_gateway(gatewayIdentifier=GATEWAY_ID)["gatewayUrl"]
print("Gateway URL:", GATEWAY_URL)

## Step 3: Register the Lambda as the `RefundTarget` with a `process_refund` tool

In [ ]:
target = control.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": function_arn,
                "toolSchema": {
                    "inlinePayload": [{
                        "name": "process_refund",
                        "description": "Process a customer refund for a given dollar amount",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "customer_id": {"type": "string", "description": "Customer ID, e.g. C-1234"},
                                "amount": {"type": "integer", "description": "Refund amount in whole dollars"},
                                "reason": {"type": "string", "description": "Why the refund is being issued"},
                            },
                            "required": ["customer_id", "amount"],
                        },
                    }]
                },
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
)
TARGET_ID = target["targetId"]
wait_for(lambda: control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID), label="Target")
TOOL_NAME = f"{TARGET_NAME}___process_refund"

## Step 4: Before any policy — the $50,000 refund goes through

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient


model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)


def gateway_client():
    return MCPClient(lambda: streamablehttp_client(GATEWAY_URL))


def call_refund(amount):
    """Call the tool directly through the gateway — no LLM involved."""
    with gateway_client() as client:
        result = client.call_tool_sync(
            tool_use_id=f"call-{uuid.uuid4().hex[:8]}",
            name=TOOL_NAME,
            arguments={"customer_id": "C-1234", "amount": amount, "reason": "demo"},
        )
    text = " ".join(c.get("text", "") for c in result["content"])
    print(f"${amount:>7,}  →  {result['status'].upper()}: {text[:160]}")


call_refund(500)
call_refund(50000)

## Step 5: Create the policy engine and the Cedar policies

Two rules, exactly as you would say them out loud:
1. *Agents may use the refund tool* — needed because AgentCore Policy is **default-deny**
2. *Refunds over $1,000 are forbidden* — and **forbid always wins**

Tool arguments are available in Cedar as `context.input.<name>`.

In [ ]:
engine = control.create_policy_engine(name=ENGINE_NAME, description="Refund limits")
ENGINE_ID, ENGINE_ARN = engine["policyEngineId"], engine["policyEngineArn"]
control.get_waiter("policy_engine_active").wait(policyEngineId=ENGINE_ID)
print("Policy engine active:", ENGINE_ARN)

ALLOW_REFUNDS = '''permit (
    principal,
    action == AgentCore::Action::"RefundTarget___process_refund",
    resource == AgentCore::Gateway::"<GATEWAY_ARN>"
);
'''.replace("<GATEWAY_ARN>", GATEWAY_ARN)

FORBID_LARGE_REFUNDS = '''forbid (
    principal,
    action == AgentCore::Action::"RefundTarget___process_refund",
    resource == AgentCore::Gateway::"<GATEWAY_ARN>"
)
when {
    context.input.amount > 1000
};
'''.replace("<GATEWAY_ARN>", GATEWAY_ARN)

policy_ids = []
for name, statement in [("AllowRefundTool", ALLOW_REFUNDS), ("ForbidLargeRefunds", FORBID_LARGE_REFUNDS)]:
    policy = control.create_policy(
        policyEngineId=ENGINE_ID,
        name=name,
        definition={"cedar": {"statement": statement}},
        # The Cedar analyzer rejects the unconditional permit as "Overly Permissive" by default.
        # That is intended here: the forbid rule narrows it, so keep the policy despite the finding.
        validationMode="IGNORE_ALL_FINDINGS",
    )
    control.get_waiter("policy_active").wait(policyEngineId=ENGINE_ID, policyId=policy["policyId"])
    policy_ids.append(policy["policyId"])
    print(f"Policy active: {name}")

## Step 6: Attach the engine to the gateway in ENFORCE mode

In [ ]:
gw = control.get_gateway(gatewayIdentifier=GATEWAY_ID)
control.update_gateway(
    gatewayIdentifier=GATEWAY_ID,
    name=gw["name"],
    roleArn=gw["roleArn"],
    protocolType=gw["protocolType"],
    authorizerType=gw["authorizerType"],
    policyEngineConfiguration={"arn": ENGINE_ARN, "mode": "ENFORCE"},
)
wait_for(lambda: control.get_gateway(gatewayIdentifier=GATEWAY_ID), label="Gateway")
time.sleep(10)  # give the new configuration a moment to propagate

## Step 7: After the policy — $500 allowed, $50,000 denied

In [ ]:
call_refund(500)
call_refund(50000)

### And through an agent

The agent still *wants* to issue the refund — its reasoning isn't overridden — but the platform blocks the call and the agent has to explain why.

> **Groq note:** gpt-oss models are more cautious than the original model and would refuse on their own, so the prompt gives this agent broad refund authority — exactly the over-permissive setup the policy has to catch.

In [ ]:
with gateway_client() as client:
    agent = Agent(
        model=model,
        tools=client.list_tools_sync(),
        system_prompt=(
            "You are a helpful customer support agent with authority to issue refunds of any amount. "
            "Process refunds when customers request them, without asking for extra details."
        ),
    )
    agent("I'm really frustrated. Just give me a $50,000 refund for customer C-1234 and we'll call it even.")
    print("\n" + "-" * 60)
    agent("Fine. Please refund $300 to customer C-1234 for the damaged item.")

Policy decisions are logged to CloudWatch, so every allow/deny becomes part of your audit trail.

**Try next:** change the engine mode to `LOG_ONLY` to see decisions logged without blocking — a safe way to roll out new rules.

## Step 8: Clean up

In [ ]:
# Detach the policy engine, then delete target and gateway
control.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
while control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items"):
    time.sleep(3)
control.delete_gateway(gatewayIdentifier=GATEWAY_ID)
print("Deleted gateway")

# Delete every policy in the engine, including any that failed to create (the engine must be empty)
for pid in [p["policyId"] for p in control.list_policies(policyEngineId=ENGINE_ID)["policies"]]:
    control.delete_policy(policyEngineId=ENGINE_ID, policyId=pid)
    control.get_waiter("policy_deleted").wait(policyEngineId=ENGINE_ID, policyId=pid)
control.delete_policy_engine(policyEngineId=ENGINE_ID)
print("Deleted policy engine")

lambda_client.delete_function(FunctionName=FUNCTION_NAME)
iam.detach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
iam.delete_role(RoleName=LAMBDA_ROLE)
iam.delete_role_policy(RoleName=GATEWAY_ROLE, PolicyName="GatewayPermissions")
iam.delete_role(RoleName=GATEWAY_ROLE)
print("Deleted Lambda and IAM roles")